## DataBunch/Learner

In [1]:
import os
import urllib.request

from pathlib import Path
from IPython.core.debugger import set_trace
# from fastai import datasets
import pickle, gzip, math, torch, matplotlib as mpl
import matplotlib.pyplot as plt
from torch import tensor

In [2]:
MNIST_URL = 'https://github.com/mnielsen/neural-networks-and-deep-learning/raw/master/data/mnist.pkl.gz'

# Set path to the local Colab content folder
path = Path('mnist.pkl.gz')

if not path.exists():
    print("Downloading MNIST pkl.gz...")
    urllib.request.urlretrieve(MNIST_URL, path)

print(f"File downloaded to: {path.absolute()}")

File downloaded to: /content/mnist.pkl.gz


In [3]:
with gzip.open(path, 'rb') as f:
    ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')

In [4]:
def get_data(path):
    # path = datasets.download_data(MNIST_URL, ext='.gz')
    with gzip.open(path, 'rb') as f:
        ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')
    return map(tensor, (x_train,y_train,x_valid,y_valid))

def normalize(x, m, s): return (x-m)/s

In [5]:
class Dataset():
    def __init__(self, x, y): self.x,self.y = x,y
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i],self.y[i]

In [6]:
import torch.nn.functional as F

In [7]:
x_train,y_train,x_valid,y_valid = get_data(path)
train_ds,valid_ds = Dataset(x_train, y_train),Dataset(x_valid, y_valid)
nh,bs = 50,64
c = y_train.max().item()+1
loss_func = F.cross_entropy

---

Factor out the connected pieces of info out of the fit() argument list

`fit(epochs, model, loss_func, opt, train_dl, valid_dl)`

Let's replace it with something that looks like this:

`fit(1, learn)`

This will allow us to tweak what's happening inside the training loop in other places of the code because the Learner object will be mutable, so changing any of its attribute elsewhere will be seen in our training loop.


do we really need all of those things passed into the fit function or can some of them be packaged up together - there's a lot of benefits to packaging up things together - when you can package up things together when they're alike things you can:
- pass them around to everything that needs them together
- create them using kind of factory methods that create them together
- do smart things, like look at the combination of them and make smart decisions for your users, rather than having to have them set everything themselves

would like to keep `epochs` but put all of the other args into a single object

firstly; `train_dl` & `valid_dl` conceptually should be 1 thing - it's the data (maybe test data also)

In [8]:
class DataBunch():
    def __init__(self, train_dl, valid_dl, c=None):
        self.train_dl,self.valid_dl,self.c = train_dl,valid_dl,c

    @property
    def train_ds(self): return self.train_dl.dataset

    @property
    def valid_ds(self): return self.valid_dl.dataset

for convenience we make it easy to grab either dataset out of here also

can use either the PyTorch DataLoader or the one made previously - same api

In [9]:
from torch.utils.data import DataLoader, SequentialSampler, RandomSampler

In [10]:
import torch.nn as nn
import torch.optim as optim

In [11]:
def get_dls(train_ds, valid_ds, bs, **kwargs):
    return (DataLoader(train_ds, batch_size=bs, shuffle=True, **kwargs),
            DataLoader(valid_ds, batch_size=bs*2, **kwargs))

In [12]:
data = DataBunch(*get_dls(train_ds, valid_ds, bs), c)

here we create a `get_model` fn which automatically sets the last layer to have the correct number of activations (`c`)
- because the data automatically knows how many activations it needs
- also optionally make it so that you can pass in c: `c=None`, so that when we then create our data we can pass in `c` which we set to our maximum y value: `c = y_train.max().item()+1`

In [13]:
def get_model(data, lr=0.5, nh=50):
    m = data.train_ds.x.shape[1]
    model = nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,data.c))
    return model, optim.SGD(model.parameters(), lr=lr)

class Learner():
    def __init__(self, model, opt, loss_func, data):
        self.model,self.opt,self.loss_func,self.data = model,opt,loss_func,data

notice: our `Learner` class has no logic at all, it's just a storage "device" for these 4 things

In [14]:
learn = Learner(*get_model(data), loss_func, data)

because `model` & `opt` are returned in this order from `get_model`, can just say: `*get_model`

nothing magic going on with `DataBunch`'s and `Learner`'s - they're just like wrappers for the info we need

same `fit` function as previously but:
- `model` -> `learn.model`
- `data` -> `learn.data`
- etc

In [15]:
def accuracy(out, yb): return (torch.argmax(out, dim=1)==yb).float().mean()

In [16]:
def fit(epochs, learn):
    for epoch in range(epochs):
        learn.model.train()
        for xb,yb in learn.data.train_dl:
            loss = learn.loss_func(learn.model(xb), yb)
            loss.backward()
            learn.opt.step()
            learn.opt.zero_grad()

        learn.model.eval()
        with torch.no_grad():
            tot_loss,tot_acc = 0.,0.
            for xb,yb in learn.data.valid_dl:
                pred = learn.model(xb)
                tot_loss += learn.loss_func(pred, yb)
                tot_acc  += accuracy (pred,yb)
        nv = len(learn.data.valid_dl)
        print(epoch, tot_loss/nv, tot_acc/nv)
    return tot_loss/nv, tot_acc/nv

In [17]:
loss,acc = fit(1, learn)

0 tensor(0.2504) tensor(0.9269)


Now let's add callbacks

## CallbackHandler
This was our training loop (without validation) from the previous notebook, with the inner loop contents factored out:
```
def one_batch(xb,yb):
    pred = model(xb)
    loss = loss_func(pred, yb)
    loss.backward()
    opt.step()
    opt.zero_grad()
    
def fit():
    for epoch in range(epochs):
        for b in train_dl: one_batch(*b)
```
Add callbacks so we can remove complexity from loop, and make it flexible:

In [18]:
def one_batch(xb, yb, cb):
    if not cb.begin_batch(xb,yb): return
    loss = cb.learn.loss_func(cb.learn.model(xb), yb)
    if not cb.after_loss(loss): return
    loss.backward()
    if cb.after_backward(): cb.learn.opt.step()
    if cb.after_step(): cb.learn.opt.zero_grad()

def all_batches(dl, cb):
    for xb,yb in dl:
        one_batch(xb, yb, cb)
        if cb.do_stop(): return

def fit(epochs, learn, cb):
    if not cb.begin_fit(learn): return
    for epoch in range(epochs):
        if not cb.begin_epoch(epoch): continue
        all_batches(learn.data.train_dl, cb)

        if cb.begin_validate():
            with torch.no_grad(): all_batches(learn.data.valid_dl, cb)
        if cb.do_stop() or not cb.after_epoch(): break
    cb.after_fit()

when looking back at what is written; there's this `cb` object (the callback handler) that's being passed everywhere
- that's a **code smell**; says something should have that state
- specifically these 3 fns should be the methods of something that has this state
  - refer to new `Runner` class below
- lot's of duplicate code: horrible **code smell**
  - code duplication = cognitive overhead to understand what's going on, lots of opportunities for errors & places to change if you need to edit something
  - factored it out into `__call__` further below

In [19]:
class Callback():
    def begin_fit(self, learn):
        self.learn = learn
        return True
    def after_fit(self): return True
    def begin_epoch(self, epoch):
        self.epoch=epoch
        return True
    def begin_validate(self): return True
    def after_epoch(self): return True
    def begin_batch(self, xb, yb):
        self.xb,self.yb = xb,yb
        return True
    def after_loss(self, loss):
        self.loss = loss
        return True
    def after_backward(self): return True
    def after_step(self): return True

the `CallbackHandler` basically goes through every callback & calls it, and keeps track of whether we've received a `False` or not (means don't continue)

In [20]:
class CallbackHandler():
    def __init__(self,cbs=None):
        self.cbs = cbs if cbs else []

    def begin_fit(self, learn):
        self.learn,self.in_train = learn,True
        learn.stop = False
        res = True
        for cb in self.cbs: res = res and cb.begin_fit(learn)
        return res

    def after_fit(self):
        res = not self.in_train
        for cb in self.cbs: res = res and cb.after_fit()
        return res

    def begin_epoch(self, epoch):
        self.learn.model.train()
        self.in_train=True
        res = True
        for cb in self.cbs: res = res and cb.begin_epoch(epoch)
        return res

    def begin_validate(self):
        self.learn.model.eval()
        self.in_train=False
        res = True
        for cb in self.cbs: res = res and cb.begin_validate()
        return res

    def after_epoch(self):
        res = True
        for cb in self.cbs: res = res and cb.after_epoch()
        return res

    def begin_batch(self, xb, yb):
        res = True
        for cb in self.cbs: res = res and cb.begin_batch(xb, yb)
        return res

    def after_loss(self, loss):
        res = self.in_train
        for cb in self.cbs: res = res and cb.after_loss(loss)
        return res

    def after_backward(self):
        res = True
        for cb in self.cbs: res = res and cb.after_backward()
        return res

    def after_step(self):
        res = True
        for cb in self.cbs: res = res and cb.after_step()
        return res

    def do_stop(self):
        try:     return self.learn.stop
        finally: self.learn.stop = False

example of a little callback we could create:
- at the start of `begin_fit` sets number of iteration to 0 (`self.n_iters = 0`)
- after every step: `self.n_iters += 1`
   - if it gets past 10 iterations, tells the learner to stop: `if self.n_iters>=10: self.learn.stop = True`
    - because the `CallbackHandler` has a `do_stop` that gets checked at the end

In [21]:
class TestCallback(Callback):
    def begin_fit(self,learn):
        super().begin_fit(learn)
        self.n_iters = 0
        return True

    def after_step(self):
        self.n_iters += 1
        print(self.n_iters)
        if self.n_iters>=10: self.learn.stop = True
        return True

In [22]:
fit(1, learn, cb=CallbackHandler([TestCallback()]))

1
2
3
4
5
6
7
8
9
10


as we can see it only did 10 batches.  
really handy if you just want to run a few batches & see if everything's working and don't want to run a whole epoch - this is a quick way to do so

This is roughly how fastai does it now (except that the handler can also change and return xb, yb, and loss). But let's see if we can make things simpler and more flexible, so that a single class has access to everything and can change anything at any time. The fact that we're passing cb to so many functions is a strong hint they should all be in the same class!

## Runner

In [23]:
import re

_camel_re1 = re.compile('(.)([A-Z][a-z]+)')
_camel_re2 = re.compile('([a-z0-9])([A-Z])')
def camel2snake(name):
    s1 = re.sub(_camel_re1, r'\1_\2', name)
    return re.sub(_camel_re2, r'\1_\2', s1).lower()

class Callback():
    _order=0
    def set_runner(self, run): self.run=run
    def __getattr__(self, k): return getattr(self.run, k)
    @property
    def name(self):
        name = re.sub(r'Callback$', '', self.__class__.__name__)
        return camel2snake(name or 'callback')

`_order=0` attribute - allows us to make sure that some things run after others
- very often you want to be able to inject behavior into something, but the different things can influence each other; eg: transformations (will cover this when we do data augmentations)
- so quite often you'll need things to run in a particular order
- so when adding this "injectable" behavior like callbacks, like to add something that controls which order it should run in.

see: `for cb in sorted(self.cbs, key=lambda x: x._order):`

`__getattr__`
- an importing thing to note about `__getattr__` is that it is only called by Python if it can't find the attribute that you've asked for
  - so here; if something asks for `self.name` we have that defined so it's never going to execute `__getattr__`
  - if you get to `return getattr(self.run, k)` it means python looked for this attribute & couldn't find it
- very often the thing you want in the caller is inside the runner (stored away as `self.run`) - this means in all of our callback you can just use `self.` for pretty much everything and it will grab what you want, even though most of the stuff is inside the runner
  - see this pattern in fastai alot: when one object contains (or composes) another object we very often delegate getattr to the other object: `def __getattr__(self, k): return getattr(self.run, k)`
    - eg: if you're looking at a dataset - delegate to x
    - stuff in the datablocks api will often delegate to stuff lower in the datablocks api, etc

---

This first callback is reponsible to switch the model back and forth in training or validation mode, as well as maintaining a count of the iterations, or the percentage of iterations ellapsed in the epoch.

In [24]:
class TrainEvalCallback(Callback):
    def begin_fit(self):
        self.run.n_epochs=0.
        self.run.n_iter=0

    def after_batch(self):
        if not self.in_train: return
        self.run.n_epochs += 1./self.iters
        self.run.n_iter   += 1

    def begin_epoch(self):
        self.run.n_epochs=self.epoch
        self.model.train()
        self.run.in_train=True

    def begin_validate(self):
        self.model.eval()
        self.run.in_train=False

We'll also re-create our TestCallback

In [30]:
class TestCallback(Callback):
    _order=1
    def after_step(self):
        if self.n_iters>=10: return True

In [26]:
cbname = 'TrainEvalCallback'
camel2snake(cbname)

'train_eval_callback'

In [27]:
TrainEvalCallback().name

'train_eval'

In [28]:
from typing import *

def listify(o):
    if o is None: return []
    if isinstance(o, list): return o
    if isinstance(o, str): return [o]
    if isinstance(o, Iterable): return list(o)
    return [o]

`Runner` class below contains the 3 fns (methods) from above:

In [29]:
class Runner():
    def __init__(self, cbs=None, cb_funcs=None):
        cbs = listify(cbs)
        for cbf in listify(cb_funcs):
            cb = cbf()
            setattr(self, cb.name, cb)
            cbs.append(cb)
        self.stop,self.cbs = False,[TrainEvalCallback()]+cbs

    @property
    def opt(self):       return self.learn.opt
    @property
    def model(self):     return self.learn.model
    @property
    def loss_func(self): return self.learn.loss_func
    @property
    def data(self):      return self.learn.data

    def one_batch(self, xb, yb):
        self.xb,self.yb = xb,yb
        if self('begin_batch'): return
        self.pred = self.model(self.xb)
        if self('after_pred'): return
        self.loss = self.loss_func(self.pred, self.yb)
        if self('after_loss') or not self.in_train: return
        self.loss.backward()
        if self('after_backward'): return
        self.opt.step()
        if self('after_step'): return
        self.opt.zero_grad()

    def all_batches(self, dl):
        self.iters = len(dl)
        for xb,yb in dl:
            if self.stop: break
            self.one_batch(xb, yb)
            self('after_batch')
        self.stop=False

    def fit(self, epochs, learn):
        self.epochs,self.learn = epochs,learn

        try:
            for cb in self.cbs: cb.set_runner(self)
            if self('begin_fit'): return
            for epoch in range(epochs):
                self.epoch = epoch
                if not self('begin_epoch'): self.all_batches(self.data.train_dl)

                with torch.no_grad():
                    if not self('begin_validate'): self.all_batches(self.data.valid_dl)
                if self('after_epoch'): break

        finally:
            self('after_fit')
            self.learn = None

    def __call__(self, cb_name):
        for cb in sorted(self.cbs, key=lambda x: x._order):
            f = getattr(cb, cb_name, None)
            if f and f(): return True
        return False

`__call__` is the smallest possible way you can call something; you don't have to give it a name at all when you call it
- eg: `self('after_epoch')` calls the callback `after_epoch`

`__getattr__` looks inside given object and tries to find something with the given name.

Third callback: how to compute metrics.

In [31]:
class AvgStats():
    def __init__(self, metrics, in_train): self.metrics,self.in_train = listify(metrics),in_train

    def reset(self):
        self.tot_loss,self.count = 0.,0
        self.tot_mets = [0.] * len(self.metrics)

    @property
    def all_stats(self): return [self.tot_loss.item()] + self.tot_mets
    @property
    def avg_stats(self): return [o/self.count for o in self.all_stats]

    def __repr__(self):
        if not self.count: return ""
        return f"{'train' if self.in_train else 'valid'}: {self.avg_stats}"

    def accumulate(self, run):
        bn = run.xb.shape[0]
        self.tot_loss += run.loss * bn
        self.count += bn
        for i,m in enumerate(self.metrics):
            self.tot_mets[i] += m(run.pred, run.yb) * bn

class AvgStatsCallback(Callback):
    def __init__(self, metrics):
        self.train_stats,self.valid_stats = AvgStats(metrics,True),AvgStats(metrics,False)

    def begin_epoch(self):
        self.train_stats.reset()
        self.valid_stats.reset()

    def after_loss(self):
        stats = self.train_stats if self.in_train else self.valid_stats
        with torch.no_grad(): stats.accumulate(self.run)

    def after_epoch(self):
        print(self.train_stats)
        print(self.valid_stats)

so that's the entirety of what it took to add metrics and loss tracking to our minimal training loop

notice: `accumulate` method fixed problem of having different batch size in the avg

`AvgStatsCallback`
- a couple of objects to keep track of our metrics
- at the start of an epoch: reset the stats
- after we've got the loss calculated: accumulate the stats
- at the end of an epoch: print out the stats

In [32]:
learn = Learner(*get_model(data), loss_func, data)

In [33]:
stats = AvgStatsCallback([accuracy])
run = Runner(cbs=stats)

In [34]:
run.fit(2, learn)

train: [0.31264103515625, tensor(0.9041)]
valid: [0.14663204345703126, tensor(0.9584)]
train: [0.14253482421875, tensor(0.9563)]
valid: [0.1407530517578125, tensor(0.9584)]


In [35]:
loss,acc = stats.valid_stats.avg_stats
assert acc>0.9
loss,acc

(0.1407530517578125, tensor(0.9584))

In [36]:
from functools import partial

In [37]:
acc_cbf = partial(AvgStatsCallback,accuracy)

In [38]:
run = Runner(cb_funcs=acc_cbf)

In [39]:
run.fit(1, learn)

train: [0.10754703125, tensor(0.9666)]
valid: [0.1185899658203125, tensor(0.9650)]


Using Jupyter means we can get tab-completion even for dynamic code like this! :)

In [40]:
run.avg_stats.valid_stats.avg_stats

[0.1185899658203125, tensor(0.9650)]